In [22]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import openml

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import mlflow
import mlflow.sklearn
import mlflow.xgboost

import dagshub

In [23]:
dagshub.init(
    repo_owner="pranav-0406",
    repo_name="Boston_ML_Flow",
    mlflow=True
)

Initialized MLflow to track repo "pranav-0406/Boston_ML_Flow"

Repository pranav-0406/Boston_ML_Flow initialized!

In [24]:
import mlflow

mlflow.set_experiment(
    "Boston Housing Linear Regression"
)

<Experiment: artifact_location='mlflow-artifacts:/10a7eb29a8be4581959e5efca448f07f', creation_time=1786006581786, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786006581786, lifecycle_stage='active', name='Boston Housing Linear Regression', tags={}, trace_location=None, workspace='default'>

In [25]:
boston = openml.datasets.get_dataset("boston")

X, y, _, _ = boston.get_data(target=boston.default_target_attribute)

print(X.shape)
print(y.shape)

(506, 13)
(506,)


In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

# One-hot encode any categorical/object columns and align test columns to train
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [27]:
print("Training Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)

Training Shape: (354, 20)
Testing Shape : (152, 20)


In [28]:
models = [

    (
        "Linear Regression",
        LinearRegression()
    ),

    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=100,
            random_state=42
        )
    ),

    (
        "XGBoost",
        XGBRegressor(
            random_state=42
        )
    )

]

In [29]:
reports = []
trained_models = []

for model_name, model in models:

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    mse = mean_squared_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        predictions
    )

    report = {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

    reports.append(report)
    trained_models.append(model)

    print("="*50)
    print(model_name)
    print("="*50)

    print(report)

Linear Regression
{'MAE': 3.174117748294079, 'MSE': 21.259848180048845, 'RMSE': np.float64(4.61084029001752), 'R2': 0.7146830631847092}
Random Forest
{'MAE': 2.079348684210526, 'MSE': 9.429977427631579, 'RMSE': np.float64(3.070826831267367), 'R2': 0.8734453674784899}
XGBoost
{'MAE': 2.063653735110634, 'MSE': 9.297587553521362, 'RMSE': np.float64(3.0491945745592166), 'R2': 0.875222100455442}


In [30]:
reports = []
trained_models = []

for model_name, model in models:

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    mse = mean_squared_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        predictions
    )

    report = {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

    reports.append(report)
    trained_models.append(model)

    print("="*50)
    print(model_name)
    print("="*50)

    print(report)

Linear Regression
{'MAE': 3.174117748294079, 'MSE': 21.259848180048845, 'RMSE': np.float64(4.61084029001752), 'R2': 0.7146830631847092}
Random Forest
{'MAE': 2.079348684210526, 'MSE': 9.429977427631579, 'RMSE': np.float64(3.070826831267367), 'R2': 0.8734453674784899}
XGBoost
{'MAE': 2.063653735110634, 'MSE': 9.297587553521362, 'RMSE': np.float64(3.0491945745592166), 'R2': 0.875222100455442}


In [31]:
for i, (model_name, model) in enumerate(models):

    report = reports[i]

    with mlflow.start_run(run_name=model_name):

        mlflow.log_param(
            "Model",
            model_name
        )

        mlflow.log_params(
            model.get_params()
        )

        mlflow.log_metric(
            "MAE",
            report["MAE"]
        )

        mlflow.log_metric(
            "MSE",
            report["MSE"]
        )

        mlflow.log_metric(
            "RMSE",
            report["RMSE"]
        )

        mlflow.log_metric(
            "R2",
            report["R2"]
        )

        if "XGBoost" in model_name:

            mlflow.xgboost.log_model(
                model,
                "model"
            )

        else:

            mlflow.sklearn.log_model(
                model,
                "model"
            )

2026/08/06 14:33:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: https://dagshub.com/pranav-0406/Boston_ML_Flow.mlflow/#/experiments/0/runs/0a502799b07348e98135568aaca65acb
🧪 View experiment at: https://dagshub.com/pranav-0406/Boston_ML_Flow.mlflow/#/experiments/0


2026/08/06 14:33:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: https://dagshub.com/pranav-0406/Boston_ML_Flow.mlflow/#/experiments/0/runs/3a3a1a8f53754d34883ec061f7e97620
🧪 View experiment at: https://dagshub.com/pranav-0406/Boston_ML_Flow.mlflow/#/experiments/0


2026/08/06 14:34:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: https://dagshub.com/pranav-0406/Boston_ML_Flow.mlflow/#/experiments/0/runs/1698e129ccce4363bc07fc35d6489c33
🧪 View experiment at: https://dagshub.com/pranav-0406/Boston_ML_Flow.mlflow/#/experiments/0


In [32]:
best_index = np.argmax(
    [
        r["R2"]
        for r in reports
    ]
)

best_model_name = models[best_index][0]

best_model = trained_models[best_index]

best_report = reports[best_index]

print("Best Model :", best_model_name)
print(best_report)

Best Model : XGBoost
{'MAE': 2.063653735110634, 'MSE': 9.297587553521362, 'RMSE': np.float64(3.0491945745592166), 'R2': 0.875222100455442}


In [33]:
with mlflow.start_run(
    run_name=f"Champion_{best_model_name}"
):

    mlflow.log_param(
        "Model",
        best_model_name
    )

    mlflow.log_metric(
        "MAE",
        best_report["MAE"]
    )

    mlflow.log_metric(
        "MSE",
        best_report["MSE"]
    )

    mlflow.log_metric(
        "RMSE",
        best_report["RMSE"]
    )

    mlflow.log_metric(
        "R2",
        best_report["R2"]
    )

    if "XGBoost" in best_model_name:

        mlflow.xgboost.log_model(
            best_model,
            "model",
            registered_model_name="Boston_Best_Model"
        )

    else:

        mlflow.sklearn.log_model(
            best_model,
            "model",
            registered_model_name="Boston_Best_Model"
        )

print("Best Model Registered Successfully")

2026/08/06 14:35:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Boston_Best_Model'.
2026/08/06 14:35:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Boston_Best_Model, version 1
Created version '1' of model 'Boston_Best_Model'.


🏃 View run Champion_XGBoost at: https://dagshub.com/pranav-0406/Boston_ML_Flow.mlflow/#/experiments/0/runs/71a9fb4e8080468286cf847c8d6b7a77
🧪 View experiment at: https://dagshub.com/pranav-0406/Boston_ML_Flow.mlflow/#/experiments/0
Best Model Registered Successfully
